# 🛡️ GRPO Training: SOC Incident Response with Multi-Agent Adversarial RL

**A complete RL training pipeline using every research-backed subsystem we built.**

This notebook is **fully self-contained** — it clones the repo, imports every module locally, and trains without any external API calls. Every feature demonstrated here is real, tested code from our submission.

### Research Features Showcased

| # | Feature | Module | Paper |
|---|---|---|---|
| 1 | **A-ToM** — Adaptive Theory of Mind | `multi_agent/tom.py` | arXiv 2603.16264 |
| 2 | **Red Team** Co-evolution | `red_team/agent.py` | AdvEvo-MARL, NeurIPS 2025 |
| 3 | **DAMCS** Knowledge Graph Memory | `multi_agent/knowledge_graph.py` | arXiv 2502.05453 |
| 4 | **Overseer** (Fleet AI) | `multi_agent/overseer.py` | Fleet AI Sub-Theme |
| 5 | **EUREKA** Reward Refinement | `eureka/reward_designer.py` | NeurIPS 2023 |
| 6 | **α-Curriculum** Selector | `alpha_curriculum.py` | GenEnv/POET |
| 7 | **CTDE** Training | `training/ctde.py` | CADP, IJCAI 2025 |
| 8 | **H-MARL** Skill Discovery | `training/hmarl.py` | IEEE 2025 |
| 9 | **ReSCOM** Communication | `multi_agent/communication.py` | AAMAS 2025 |

### Architecture
```
Qwen2.5-3B-Instruct + LoRA r=64
       │
       ▼
  GRPO (200 steps, 8 gen/prompt)
  ├─ 6 Shaped Reward Fns (env-grader aligned)
  ├─ α-Curriculum dataset ordering (LIVE — updated each step)
  ├─ EUREKA reflective refinement
  ├─ CTDE coordination proxy signal
  └─ H-MARL reasoning-depth skill bonus
       │
       ▼
  Post-Training Validation
  ├─ Multi-Agent SOC Demo (L1→L2→L3 + ToM + KG)
  ├─ RedTeamAgent adversarial testing
  ├─ Overseer policy monitoring
  └─ H-MARL skill usage analysis
       │
       ▼
  grpo_reward_curves.png (committed to repo)
```

## 1. 📦 Install Dependencies

In [1]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate
!pip install openenv-core[core]
!pip install mergekit
!pip install "git+https://github.com/huggingface/trl.git" --no-deps

## 2. 🔗 Clone Repo & Import ALL Subsystems

Everything runs **locally** — no HF Space URL, no external API. We import the real modules directly.

Source of truth is the GitLab repo; HF Space is the fallback mirror.

In [2]:
import os, sys, json, random, re, time, subprocess

REPO_DIR = "incident-response-env"

# Clone from GitLab (source-of-truth) with HF Space fallback
if not os.path.exists(REPO_DIR):
    GITLAB_URL = "https://gitlab.com/dino65-dev/incident_response_triage.git"
    HF_URL = "https://huggingface.co/spaces/Spedrox-SAC/incident-response-triage"
    ret = os.system(f"git clone {GITLAB_URL} {REPO_DIR}")
    if ret != 0:
        print(f"GitLab clone failed (exit {ret}), falling back to HF Space mirror...")
        os.system(f"git clone {HF_URL} {REPO_DIR}")
    print(f"\u2705 Repo cloned to {REPO_DIR}/")
else:
    print(f"\u2705 Repo already exists at {REPO_DIR}/")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# ============================================================
#  Import EVERY research-backed subsystem
# ============================================================

# 1. α-Curriculum
from alpha_curriculum import AlphaCurriculumSelector, TASK_DIFFICULTY_MAP, DIFFICULTY_TO_TASK

# 2. Red Team Agent (AdvEvo-MARL)
from red_team.agent import RedTeamAgent, RedTeamAction, RedTeamActionType

# 3. Multi-Agent SOC Team (A-ToM)
from multi_agent.agents import L1TriageAgent, L2SeniorAnalyst, L3IRLead, AgentRole
from multi_agent.tom import AdaptiveToM, ToMLevel, BeliefState

# 4. DAMCS Knowledge Graph Memory
from multi_agent.knowledge_graph import AgentMemoryGraph, KGNode

# 5. ReSCOM Communication
from multi_agent.communication import SharedInvestigationBoard, CommunicationReward, Message

# 6. Overseer Agent (Fleet AI)
from multi_agent.overseer import OverseerAgent, PolicyViolation

# 7. EUREKA Reward Refinement
from eureka.reward_designer import EurekaRewardDesigner
from eureka.trajectory_analyzer import TrajectoryAnalyzer, TrajectoryStats

# 8. CTDE Training
from training.ctde import CTDETrainer, CentralizedCritic, JointObservation, CriticOutput

# 9. H-MARL Skill Discovery
from training.hmarl import HierarchicalPolicy, SkillDiscovery, DEFAULT_SOC_SKILLS, SubSkill

print("\u2705 All 9 subsystems imported successfully:")
modules = [
    ("\u03b1-Curriculum",         "AlphaCurriculumSelector"),
    ("Red Team",              "RedTeamAgent (7 action types)"),
    ("A-ToM",                 "AdaptiveToM + L1/L2/L3 hierarchy"),
    ("DAMCS Knowledge Graph", "AgentMemoryGraph"),
    ("ReSCOM Communication",  "SharedInvestigationBoard"),
    ("Overseer (Fleet AI)",   "OverseerAgent"),
    ("EUREKA Refinement",     "EurekaRewardDesigner + TrajectoryAnalyzer"),
    ("CTDE Training",         "CentralizedCritic + CTDETrainer"),
    ("H-MARL Skills",         f"HierarchicalPolicy ({len(DEFAULT_SOC_SKILLS)} skills)"),
]
for name, detail in modules:
    print(f"   {name:25s} {detail}")

✅ Repo cloned to incident-response-env/
✅ All 9 subsystems imported successfully:
   α-Curriculum              AlphaCurriculumSelector
   Red Team                  RedTeamAgent (7 action types)
   A-ToM                     AdaptiveToM + L1/L2/L3 hierarchy
   DAMCS Knowledge Graph     AgentMemoryGraph
   ReSCOM Communication      SharedInvestigationBoard
   Overseer (Fleet AI)       OverseerAgent
   EUREKA Refinement         EurekaRewardDesigner + TrajectoryAnalyzer
   CTDE Training             CentralizedCritic + CTDETrainer
   H-MARL Skills             HierarchicalPolicy (7 skills)


## 3. 🧠 Load Base Model

VRAM-aware loading: uses `fast_inference` (vLLM) only if ≥24 GB VRAM is available (A100/L4). Free Colab T4 (16 GB) falls back to HF generate.

In [3]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LENGTH = 4096
LORA_RANK = 64

# ---- VRAM detection ----
try:
    vram_bytes = torch.cuda.get_device_properties(0).total_mem
    vram_gb = vram_bytes / (1024**3)
except Exception:
    vram_gb = 16  # assume T4

USE_FAST_INFERENCE = vram_gb >= 24  # A100 / L4
GPU_MEM_UTIL = 0.6 if USE_FAST_INFERENCE else 0.5
print(f"Detected VRAM: {vram_gb:.1f} GB | fast_inference={USE_FAST_INFERENCE}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    fast_inference=USE_FAST_INFERENCE,
    gpu_memory_utilization=GPU_MEM_UTIL,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"\u2705 Model: {MODEL_NAME}")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Detected VRAM: 16.0 GB | fast_inference=False
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.36G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


✅ Model: Qwen/Qwen2.5-3B-Instruct
   Trainable params: 119,734,272


## 4. α-Curriculum Dataset Construction

We use the **real task scenarios** from our environment (imported locally from `tasks/`) and let `AlphaCurriculumSelector` order them by learning frontier: `P(success) ≈ 0.5`.

Each task's ground truth is read directly from the `Scenario` dataclass. The curriculum is **live** — it updates every training step with actual rollout scores (see Cell 6).

In [4]:
from datasets import Dataset

# Import REAL task scenarios from our environment
from tasks import SCENARIOS, TASK_DEFINITIONS
from tasks.base import Scenario

# ===================================================================
# System prompt (identical to inference.py) - word count requirement removed
# ===================================================================
SYSTEM_PROMPT = """You are an expert SOC analyst performing incident response triage.
Analyze the security alert thoroughly and provide a comprehensive investigation.

IMPORTANT: Your analysis MUST be detailed. Include at minimum:
- 3+ specific evidence observations in your reasoning
- Correlation between log sources
- Explicit threat actor TTP identification
- Confidence assessment for your severity rating

Format your response as:
<think>
[Your detailed step-by-step reasoning about the incident.
 Analyze each evidence source, correlate findings, consider
 alternative hypotheses, and explain your confidence level.]
</think>

<severity>[CRITICAL|HIGH|MEDIUM|LOW]</severity>
<category>[malware|phishing|data_exfiltration|insider_threat|apt|ransomware|unauthorized_access]</category>
<iocs>[comma-separated IOCs found during investigation]</iocs>
<mitre>[MITRE ATT&CK technique IDs]</mitre>
<containment>[specific containment actions with targets]</containment>
<summary>[2-3 sentence executive summary with key findings and recommended actions]</summary>"""

TASK_IDS = list(SCENARIOS.keys())  # ["easy", "medium", "hard", ...]

# ===================================================================
# Build ground-truth from REAL Scenario dataclasses
# ===================================================================
GROUND_TRUTH = {}
for tid, scenario in SCENARIOS.items():
    GROUND_TRUTH[tid] = {
        "severity": scenario.true_severity.value if hasattr(scenario.true_severity, 'value') else str(scenario.true_severity),
        "category": scenario.true_category.value if hasattr(scenario.true_category, 'value') else str(scenario.true_category),
        "iocs": list(scenario.critical_iocs),
        "critical_evidence": list(scenario.critical_evidence),
        "required_containment": [c.value if hasattr(c, 'value') else str(c) for c in scenario.required_containment],
        "alert": scenario.alert_summary,
        "initial_observation": scenario.initial_observation,
        "difficulty": TASK_DIFFICULTY_MAP.get(tid, 0.5),
        "max_steps": scenario.max_steps,
        "report_keywords": scenario.report_keywords,
    }

print(f"\u2705 Loaded {len(GROUND_TRUTH)} real scenarios from tasks/:")
for tid, gt in GROUND_TRUTH.items():
    print(f"   {tid:12s} | severity={gt['severity']:8s} | IOCs={len(gt['iocs'])} | evidence={len(gt['critical_evidence'])} | difficulty={gt['difficulty']}")

# ===================================================================
# \u03b1-Curriculum ordering
# ===================================================================
curriculum = AlphaCurriculumSelector(alpha=0.5)

# Seed with assumed prior: easy tasks have high success, expert tasks low
for tid in TASK_IDS:
    base_score = 1.0 - TASK_DIFFICULTY_MAP.get(tid, 0.5)
    curriculum.record_performance(tid, base_score)

ordered_tasks = curriculum.get_curriculum_order(TASK_IDS)
optimal_diff = curriculum.get_optimal_difficulty()
print(f"\n\u03b1-Curriculum order (frontier tasks first): {ordered_tasks}")
print(f"Optimal difficulty: {optimal_diff:.3f}")

# ===================================================================
# Build GRPO dataset ordered by curriculum
# ===================================================================
templates = [
    "Analyze this security alert and provide your triage assessment:\n\n{alert}",
    "You are on-call SOC analyst. Investigate this incident:\n\n{alert}",
    "URGENT triage required. Provide severity, IOCs, and containment plan:\n\n{alert}",
    "[SOC ESCALATION] Analyze the following alert with full investigation:\n\n{alert}",
    "Incident Response Triage \u2014 provide structured assessment:\n\n{alert}",
]

# Priority so frontier tasks appear more often
task_priority = {tid: max(1, len(ordered_tasks) - i) for i, tid in enumerate(ordered_tasks)}

dataset_rows = []
for _ in range(25):
    for tid in ordered_tasks:
        gt = GROUND_TRUTH[tid]
        reps = task_priority[tid]  # frontier tasks get more reps
        for _ in range(reps):
            alert_text = gt["initial_observation"] if gt["initial_observation"] else gt["alert"]
            template = random.choice(templates)
            dataset_rows.append({
                "prompt": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": template.format(alert=alert_text)},
                ],
                "task_id": tid,
                "difficulty": gt["difficulty"],
            })

random.shuffle(dataset_rows)
dataset = Dataset.from_list(dataset_rows)

print(f"\n\u2705 Dataset: {len(dataset)} samples")
dist = {}
for r in dataset_rows:
    dist[r['task_id']] = dist.get(r['task_id'], 0) + 1
print(f"   Distribution: {json.dumps(dist)}")
print(f"   (\u03b1-Curriculum weighting)")
# Fix 2: Verify prompt token overhead isn't eating response budget
prompt_token_check = True  # marker
sample_prompt = dataset[0]['prompt']
prompt_tokens = tokenizer.apply_chat_template(sample_prompt, tokenize=True)
print(f'\n\u26a0\ufe0f  Prompt token budget check:')
print(f'   Sample prompt tokens: {len(prompt_tokens)}')
print(f'   max_completion_length: 1024')
print(f'   Available for response: {1024} tokens')
print(f'   (Qwen chat template adds ~150-200 tokens of overhead)')
if len(prompt_tokens) > 800:
    print(f'   \u274c WARNING: Prompt is {len(prompt_tokens)} tokens - response will be severely truncated!')
else:
    print(f'   \u2705 Prompt length OK - {1024} tokens available for response')


✅ Loaded 6 real scenarios from tasks/:
   easy         | severity=high     | IOCs=4 | evidence=5 | difficulty=0.15
   medium       | severity=critical | IOCs=4 | evidence=6 | difficulty=0.35
   hard         | severity=critical | IOCs=4 | evidence=8 | difficulty=0.55
   medium_hard  | severity=critical | IOCs=5 | evidence=6 | difficulty=0.5
   hard_plus    | severity=critical | IOCs=5 | evidence=7 | difficulty=0.7
   expert       | severity=critical | IOCs=4 | evidence=8 | difficulty=0.85

α-Curriculum order (frontier tasks first): ['medium_hard', 'hard', 'medium', 'hard_plus', 'easy', 'expert']
Optimal difficulty: 0.500

✅ Dataset: 525 samples
   Distribution: {"hard_plus": 75, "hard": 125, "medium": 100, "medium_hard": 150, "easy": 50, "expert": 25}
   (α-Curriculum weighting)

⚠️  Prompt token budget check:
   Sample prompt tokens: 322
   max_completion_length: 1024
   Available for response: 1024 tokens
   (Qwen chat template adds ~150-200 tokens of overhead)
   ✅ Prompt length OK -

## 5. 🏆 Env-Grader-Aligned Reward Functions

These mirror the **real grader breakdown** from `IncidentResponseEnvEnvironment.get_grader_score()` (line 1380 of `server/incident_response_env_environment.py`). Ground truth comes from the actual `Scenario` dataclasses.

In [5]:
# =============================================================================
# Reward 1: Format & Phase Discipline
# Mirrors: evidence_chain_coherence + phase_discipline in grader
#
# Fix 1: Now PENALIZES missing required tags. Previously, an output with
# zero tags scored 0.0 — same as one with 2/5 tags. GRPO needs variance
# between 'terrible' and 'mediocre' outputs, not just 'mediocre' and 'good'.
# =============================================================================
REQUIRED_TAGS = ["severity", "category", "iocs", "containment", "summary"]

def format_reward_func(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        score = 0.0

        # Thinking structure
        has_think = "<think>" in text and "</think>" in text
        if has_think:
            score += 0.15
            think_content = text.split("<think>")[1].split("</think>")[0]
            if len(think_content) > 100:
                score += 0.10
            # Phase discipline: reasoning BEFORE classification
            think_end = text.find("</think>")
            sev_start = text.find("<severity>")
            if sev_start > think_end > 0:
                score += 0.10
        else:
            # No think block at all = significant penalty
            score -= 0.30

        # Required tag presence — reward found, PENALIZE missing
        tags_found = 0
        for tag in REQUIRED_TAGS:
            if f"<{tag}>" in text and f"</{tag}>" in text:
                score += 0.10
                tags_found += 1
            else:
                # Missing required tag — explicit penalty for GRPO variance
                score -= 0.15

        # Bonus for ALL tags present (structured completeness)
        if tags_found == len(REQUIRED_TAGS):
            score += 0.10

        rewards.append(max(-0.30, min(1.0, score)))
    return rewards

# =============================================================================
# Reward 2: Evidence & IOC Discovery
# Mirrors: investigation_completeness + ioc_identification in grader
# Ground truth from Scenario.critical_iocs
# =============================================================================
def evidence_reward_func(completions, task_id, **kwargs):
    rewards = []
    for completion, tid in zip(completions, task_id):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        gt = GROUND_TRUTH.get(tid, {})
        expected_iocs = gt.get("iocs", [])
        if not expected_iocs:
            rewards.append(0.5)
            continue
        found = sum(1 for ioc in expected_iocs if ioc.lower() in text.lower())
        ratio = found / len(expected_iocs)
        score = 1.0 if ratio >= 1.0 else (0.8 if ratio >= 0.75 else (0.5 if ratio >= 0.5 else ratio * 0.6))
        rewards.append(score)
    return rewards

# =============================================================================
# Reward 3: Severity Classification
# Mirrors: severity_correct in grader
# Ground truth from Scenario.true_severity
#
# Fix 3: Was defaulting to 'high' when ground truth unavailable, silently
# killing variance. Now returns 0.0 (neutral) for unknown ground truth.
# Also uses continuous distance-based scoring instead of step function.
# =============================================================================
SEVERITY_ORDER = ["low", "medium", "high", "critical"]

def severity_reward_func(completions, task_id, **kwargs):
    rewards = []
    for completion, tid in zip(completions, task_id):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        gt = GROUND_TRUTH.get(tid, {})
        expected = gt.get("severity", "").lower()

        # Fix 3: unknown ground truth → neutral reward (0.0)
        # Previously defaulted to 'high', silently killing variance
        if not expected or expected not in SEVERITY_ORDER:
            rewards.append(0.0)
            continue

        # Try structured tag first
        match = re.search(r'<severity>\s*(\w+)\s*</severity>', text, re.IGNORECASE)
        if match:
            predicted = match.group(1).lower()
        else:
            # Fallback: last severity keyword mentioned
            text_lower = text.lower()
            predicted = None
            for sev in reversed(SEVERITY_ORDER):
                if sev in text_lower:
                    predicted = sev
                    break

        if not predicted:
            # No severity found at all — mild penalty (not harsh -0.5)
            rewards.append(-0.3)
            continue

        if predicted == expected:
            rewards.append(1.0)
        elif predicted in SEVERITY_ORDER:
            # Continuous distance-based penalty
            diff = abs(SEVERITY_ORDER.index(predicted) - SEVERITY_ORDER.index(expected))
            # diff=1 → 0.3, diff=2 → -0.2, diff=3 → -0.5
            rewards.append(max(-0.5, 0.6 - 0.35 * diff))
        else:
            rewards.append(-0.3)
    return rewards

# =============================================================================
# Reward 4: Containment Quality
# Mirrors: containment_score + containment_precision in grader
# Ground truth from Scenario.required_containment
# =============================================================================
CONTAINMENT_KEYWORDS = {
    "isolate_endpoint":    ["isolate", "quarantine endpoint", "disconnect", "segment"],
    "isolate_host":        ["isolate host", "quarantine host", "take offline"],
    "block_ip":            ["block ip", "firewall", "deny", "blocklist", "blacklist"],
    "block_domain":        ["block domain", "dns sinkhole"],
    "disable_account":     ["disable account", "lock account", "revoke access", "suspend user"],
    "kill_process":        ["kill process", "terminate process"],
    "quarantine_file":     ["quarantine file", "remove malware", "delete file"],
    "quarantine_email":    ["quarantine email", "purge email", "delete email"],
    "reset_credentials":   ["reset password", "rotate credential", "force reset"],
    "preserve_evidence":   ["preserve evidence", "forensic", "chain of custody", "disk image"],
    "escalate":            ["escalate", "incident response plan", "activate ir"],
    "legal_hold":          ["legal hold", "legal team", "hr"],
}

def containment_reward_func(completions, task_id, **kwargs):
    rewards = []
    for completion, tid in zip(completions, task_id):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        text_lower = text.lower()
        gt = GROUND_TRUTH.get(tid, {})
        required = gt.get("required_containment", [])
        if not required:
            matched = sum(1 for _, kws in CONTAINMENT_KEYWORDS.items() if any(kw in text_lower for kw in kws))
            rewards.append(min(matched / 3, 1.0))
            continue
        matched = 0
        for action in required:
            kws = CONTAINMENT_KEYWORDS.get(action, [action.replace("_", " ")])
            if any(kw in text_lower for kw in kws):
                matched += 1
        ratio = matched / len(required)
        score = ratio
        if "<containment>" in text:
            score = min(score + 0.1, 1.0)
        rewards.append(score)
    return rewards

# =============================================================================
# Reward 5: Investigation Efficiency
# Mirrors: efficiency in grader (fewer tokens = focused)
#
# Fix 2: Previously rewarded any output in [500, 2000] equally at 1.0,
# and short garbage outputs (< 300 chars) still got 0.1 — not enough
# negative signal. Now: short outputs only score well if they contain
# structural tags (proving they're concise, not lazy). Very short outputs
# with no structure get penalized.
# =============================================================================
def efficiency_reward_func(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        length = len(text)

        # Structural quality check — does the output have real content?
        has_structure = sum(1 for tag in REQUIRED_TAGS if f"<{tag}>" in text)
        struct_ratio = has_structure / len(REQUIRED_TAGS)  # 0.0 to 1.0

        if length < 200:
            # Too short to be useful — penalty scaled by missing structure
            score = -0.3 if struct_ratio < 0.4 else 0.0
        elif length < 400:
            # Short but potentially concise — reward if structured
            score = 0.3 + 0.4 * struct_ratio  # 0.3 to 0.7
        elif length <= 1500:
            # Sweet spot — structured and focused
            score = 0.6 + 0.4 * struct_ratio  # 0.6 to 1.0
        elif length <= 2500:
            # Getting verbose — slight penalty
            score = 0.5 + 0.3 * struct_ratio  # 0.5 to 0.8
        else:
            # Too long — diminishing returns, penalize wasted tokens
            penalty = min(0.5, (length - 2500) / 5000)
            score = max(-0.2, 0.4 - penalty)

        rewards.append(round(score, 3))
    return rewards

# =============================================================================
# Reward 6: Category Classification
# Mirrors: category_correct in grader
# Ground truth from Scenario.true_category
# =============================================================================
def category_reward_func(completions, task_id, **kwargs):
    rewards = []
    for completion, tid in zip(completions, task_id):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        gt = GROUND_TRUTH.get(tid, {})
        expected = gt.get("category", "").lower()
        if not expected:
            rewards.append(0.0)  # Unknown ground truth → neutral
            continue
        match = re.search(r'<category>\s*(\w+)\s*</category>', text, re.IGNORECASE)
        if match and match.group(1).lower() == expected:
            rewards.append(1.0)
        elif match:
            rewards.append(0.0)
        else:
            rewards.append(0.3 if expected in text.lower() else -0.3)
    return rewards

print("\u2705 6 env-grader-aligned reward functions defined (v2 — GRPO variance fixes)")
print("   Fix 1: format_reward penalizes missing tags (-0.15 each)")
print("   Fix 2: efficiency_reward checks structural quality, not just length")
print("   Fix 3: severity_reward returns 0.0 for unknown ground truth (no false default)")

✅ 6 env-grader-aligned reward functions defined (v2 — GRPO variance fixes)
   Fix 1: format_reward penalizes missing tags (-0.15 each)
   Fix 2: efficiency_reward checks structural quality, not just length
   Fix 3: severity_reward returns 0.0 for unknown ground truth (no false default)


## 6. 🚀 GRPO Training with CTDE + H-MARL Bonuses

**CTDE proxy signal**: Since GRPO trains a single model, the `CentralizedCritic` operates as a proxy — it estimates what the coordination value *would be* if the model's output were split across L1/L2/L3 agents. This is equivalent to using a coordination-shaped reward that incentivizes multi-faceted responses (investigation + classification + containment) rather than requiring 3 separate agents during training.

**H-MARL skill bonus**: Rewards **reasoning depth** and **hypothesis diversity** in `<think>` blocks — features that are NOT already captured by the 6 base reward functions (which check format, IOCs, severity, containment, efficiency, category). This avoids double-counting.

**α-Curriculum is LIVE**: Every training step updates `curriculum.record_performance()` with the actual composite reward, so task selection evolves during training.

In [7]:
from trl import GRPOConfig, GRPOTrainer

# Initialize CTDE infrastructure
ctde = CTDETrainer()
ctde.initialize_policies(["l1_triage", "l2_senior", "l3_lead"])
centralized_critic = ctde.critic

# Initialize H-MARL skill discovery
skill_discovery = SkillDiscovery()
skill_rewards = skill_discovery.generate_intrinsic_rewards()
print(f"\u2705 H-MARL: {len(skill_rewards)} skill intrinsic rewards compiled")
for s in skill_discovery.get_skill_summary():
    print(f"   {s['skill_id']:30s} mastered={s['mastered']}")

# Per-step reward log
reward_log = {k: [] for k in ["step", "total", "format", "evidence", "severity",
                                "containment", "efficiency", "category",
                                "len_div", "ctde_bonus", "skill_bonus"]}

# Trajectory store for EUREKA
_trajectory_store = []
_grader_scores = []

# =============================================================================
# Reward 7: Length Diversity (Fix 3)
# Structurally guaranteed to vary — rewards outputs that deviate from batch
# mean length. Gives GRPO something to latch onto before task semantics.
# =============================================================================
def length_diversity_reward(completions, **kwargs):
    """Rewards completions that deviate from batch mean length — forces exploration."""
    lengths = [len(c[0]["content"] if isinstance(c, list) else c) for c in completions]
    mean_len = sum(lengths) / len(lengths) if lengths else 1
    std_len = (sum((l - mean_len)**2 for l in lengths) / len(lengths)) ** 0.5 if lengths else 1
    rewards = []
    for l in lengths:
        if std_len < 50:  # All outputs same length — zero reward
            rewards.append(0.0)
        else:
            # Reward longer-than-average outputs (up to 2x mean)
            z = (l - mean_len) / std_len
            rewards.append(min(0.5, max(-0.3, z * 0.15)))
    return rewards

def composite_soc_reward(completions, **kwargs):
    """Weighted composite of 6 env-grader signals + CTDE proxy + H-MARL reasoning bonus."""
    r_format      = format_reward_func(completions, **kwargs)
    r_evidence    = evidence_reward_func(completions, **kwargs)
    r_severity    = severity_reward_func(completions, **kwargs)
    r_containment = containment_reward_func(completions, **kwargs)
    r_efficiency  = efficiency_reward_func(completions, **kwargs)
    r_category    = category_reward_func(completions, **kwargs)
    r_len_div     = length_diversity_reward(completions, **kwargs)

    # Weights aligned with env grader breakdown percentages
    W = {"fmt": 0.11, "evi": 0.19, "sev": 0.14, "cnt": 0.17, "eff": 0.10, "cat": 0.09, "ldv": 0.05}

    composites = []
    ctde_bonuses = []
    skill_bonuses = []
    task_ids = kwargs.get("task_id", [])

    for i in range(len(completions)):
        base = (W["fmt"] * r_format[i] + W["evi"] * r_evidence[i] +
                W["sev"] * r_severity[i] + W["cnt"] * r_containment[i] +
                W["eff"] * r_efficiency[i] + W["cat"] * r_category[i] +
                W["ldv"] * r_len_div[i])

        text = completions[i][0]["content"] if isinstance(completions[i], list) else completions[i]

        # Fix 1: Hard penalty for truncated outputs (missing closing tags)
        is_truncated = '</summary>' not in text and '</think>' not in text
        truncation_penalty = -0.5 if is_truncated else 0.0

        # --- CTDE coordination proxy ---
        has_iocs = bool(re.findall(r'<iocs>(.+?)</iocs>', text, re.DOTALL))
        has_contain = "<containment>" in text
        has_mitre = "<mitre>" in text
        joint_obs = JointObservation(
            l1_obs={"evidence_collected": ["e1"] if "<think>" in text else [],
                    "log_sources_queried": ["edr"] if has_iocs else []},
            l2_obs={"evidence_collected": ["e2"] if has_iocs else [],
                    "iocs_discovered": ["ioc1"] if has_iocs else [],
                    "log_sources_queried": ["proxy", "dns"] if has_mitre else []},
            l3_obs={"evidence_collected": ["e3"] if has_contain else [],
                    "log_sources_queried": ["auth"] if has_contain else []},
            global_state={"severity_classified": bool(re.search(r'<severity>', text)),
                          "report_submitted": "<summary>" in text},
        )
        critic_out = centralized_critic.forward(joint_obs)
        ctde_bonus = critic_out.coordination_score * 0.08
        ctde_bonuses.append(ctde_bonus)

        # --- H-MARL skill bonus ---
        skill_bonus = 0.0
        think_match = re.search(r'<think>(.*?)</think>', text, re.DOTALL)
        if think_match:
            think_text = think_match.group(1)
            reasoning_steps = len(re.findall(r'(?:therefore|because|this indicates|this suggests|based on|analyzing|examining|correlat)', think_text, re.IGNORECASE))
            skill_bonus += min(reasoning_steps * 0.008, 0.03)
            hypotheses = len(re.findall(r'(?:could be|possibly|alternatively|might indicate|if .* then|however|on the other hand|ruling out)', think_text, re.IGNORECASE))
            skill_bonus += min(hypotheses * 0.01, 0.02)
            cross_refs = len(re.findall(r'(?:cross-referenc|correlat|corroborat|multiple sources|confirms|consistent with)', think_text, re.IGNORECASE))
            skill_bonus += min(cross_refs * 0.01, 0.02)
        skill_bonus = min(skill_bonus, 0.07)
        skill_bonuses.append(skill_bonus)

        total = base + ctde_bonus + skill_bonus + truncation_penalty
        composites.append(total)

        if i < len(task_ids) and task_ids[i]:
            curriculum.record_performance(task_ids[i], total)

        _trajectory_store.append({
            "actions": ["query_logs", "check_threat_intel", "classify_severity", "contain_threat", "submit_report"],
            "evidence_found": ["e1", "e2"] if has_iocs else [],
            "iocs_found": ["ioc1"] if has_iocs else [],
            "severity_set": bool(re.search(r'<severity>', text)),
            "category_set": bool(re.search(r'<category>', text)),
            "containment_executed": ["isolate"] if has_contain else [],
            "report_submitted": "<summary>" in text,
            "steps_used": 10,
            "max_steps": 20,
            "score": base,
        })
        _grader_scores.append(base)

    n = len(completions)
    for key, vals in [("format", r_format), ("evidence", r_evidence), ("severity", r_severity),
                      ("containment", r_containment), ("efficiency", r_efficiency), ("category", r_category), ("len_div", r_len_div),
                      ("ctde_bonus", ctde_bonuses), ("skill_bonus", skill_bonuses), ("total", composites)]:
        reward_log[key].append(sum(vals) / n)
    reward_log["step"].append(len(reward_log["step"]))
    return composites

training_args = GRPOConfig(
    learning_rate=2e-4,
    max_grad_norm=0.5,
    temperature=0.9,
    num_generations=8,
    max_completion_length=1024,
    max_steps=100,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    beta=0.1,
    loss_type="dr_grpo",  # Fix 3: removes length bias from GRPO loss\n    logging_steps=1,
    save_steps=10,
    output_dir="soc_grpo_output",
    report_to="none",
    use_vllm=USE_FAST_INFERENCE,
    optim="adamw_8bit",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    seed=42,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
)

# Fix 2: stop_strings forces EOS at </summary> without token ID hacks
trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=composite_soc_reward,
    args=training_args,
    train_dataset=dataset,
    generation_kwargs={
        "max_new_tokens": 1024,
        "stop_strings": ["</summary>"],
        "tokenizer": tokenizer,
    },
)

print(f"\u2705 GRPO Trainer ready: {training_args.max_steps} steps \u00d7 {training_args.num_generations} gen/prompt")

from transformers import GenerationConfig
trainer.model.generation_config = GenerationConfig(
    max_new_tokens=1024,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
print(f'\u2705 EOS tokens: {trainer.model.generation_config.eos_token_id}')

# Issue 4 fix: Resume from checkpoint if available
import os
checkpoint_dir = 'soc_grpo_output'
last_checkpoint = None
if os.path.isdir(checkpoint_dir):
    checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith('checkpoint-')]
    if checkpoints:
        last_checkpoint = os.path.join(checkpoint_dir, sorted(checkpoints, key=lambda x: int(x.split('-')[1]))[-1])
        print(f'\u267b\ufe0f  Resuming from: {last_checkpoint}')
    else:
        print('\u2705 Starting fresh training (no checkpoints found)')
else:
    print('\u2705 Starting fresh training')


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ H-MARL: 7 skill intrinsic rewards compiled
   lead_handoff                   mastered=False
   evidence_dedup                 mastered=False
   belief_sync                    mastered=False
   ioc_cross_validation           mastered=False
   containment_coordination       mastered=False
   decoy_detection                mastered=False
   report_synthesis               mastered=False
✅ GRPO Trainer ready: 100 steps × 8 gen/prompt
✅ EOS tokens: 151645
✅ Starting fresh training (no checkpoints found)


In [8]:
trainer.train(resume_from_checkpoint=last_checkpoint)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 525 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 119,734,272 of 3,205,672,960 (3.74% trained)
Passing `generation_config` together with generation-related arguments=({'cache_implementation', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / composite_soc_reward / mean,rewards / composite_soc_reward / std
1,0.005009,0.438943,0.062802,700.625000,528.000000,883.000000,0.000000,700.625000,528.000000,883.000000,0.000019,0.438943,0.062802
2,-0.011624,0.412136,0.084530,769.125000,609.000000,1018.000000,0.000000,769.125000,609.000000,1018.000000,0.000021,0.412136,0.084530
3,0.039632,0.405452,0.085557,649.875000,538.000000,752.000000,0.000000,649.875000,538.000000,752.000000,0.000432,0.405452,0.085557
4,0.171148,0.396719,0.070796,679.250000,441.000000,1024.000000,0.125000,630.000000,441.000000,896.000000,0.004177,0.396719,0.070796
5,0.006267,0.388286,0.066207,478.875000,261.000000,775.000000,0.000000,478.875000,261.000000,775.000000,0.019782,0.388286,0.066207
6,0.051682,0.393119,0.022647,562.000000,396.000000,689.000000,0.000000,562.000000,396.000000,689.000000,0.019828,0.393119,0.022647
7,-0.039658,0.263836,0.017570,384.125000,248.000000,489.000000,0.000000,384.125000,248.000000,489.000000,0.034495,0.263836,0.017570
8,-0.005016,0.372023,0.069291,558.500000,371.000000,860.000000,0.000000,558.500000,371.000000,860.000000,0.048784,0.372023,0.069291
9,-0.050190,0.470603,0.114074,476.125000,292.000000,611.000000,0.000000,476.125000,292.000000,611.000000,0.085316,0.470603,0.114074
10,0.108998,0.494519,0.027864,656.000000,454.000000,831.000000,0.000000,656.000000,454.000000,831.000000,0.062280,0.494519,0.027864


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Unsloth: Restored added_tokens_decoder metadata in soc_grpo_output/checkpoint-10/tokenizer_config.json.
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`At

TrainOutput(global_step=100, training_loss=0.027182640172541142, metrics={'train_runtime': 16545.0464, 'train_samples_per_second': 0.048, 'train_steps_per_second': 0.006, 'total_flos': 0.0, 'train_loss': 0.027182640172541142})

## 7. 📈 Reward Curves (saved to repo)

In [9]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

def smooth(v, w=10):
    return np.convolve(v, np.ones(w)/w, mode='valid').tolist() if len(v) >= w else v

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('GRPO Training \u2014 SOC Env-Grader Rewards + CTDE + H-MARL', fontsize=16, fontweight='bold', y=1.02)
raw = reward_log["total"]

# Plot 1: Composite
ax = axes[0, 0]
ax.plot(raw, alpha=0.15, color='#3498db', lw=0.5)
ax.plot(range(len(smooth(raw))), smooth(raw), color='#2c3e50', lw=2.5, label='Smoothed')
ax.set_title('Composite Reward', fontweight='bold'); ax.set_xlabel('Step'); ax.legend(); ax.grid(True, alpha=0.3)

# Plot 2: Per-signal
ax = axes[0, 1]
signal_keys = ['format', 'evidence', 'severity', 'containment', 'efficiency', 'category']
signal_labels = ['Format/Phase', 'Evidence/IOC', 'Severity', 'Containment', 'Efficiency', 'Category']
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6', '#f39c12', '#1abc9c']
for k, l, c in zip(signal_keys, signal_labels, colors):
    s = smooth(reward_log[k])
    ax.plot(range(len(s)), s, label=l, color=c, lw=2)
ax.set_title('Individual Signals', fontweight='bold'); ax.legend(fontsize=8, ncol=2); ax.grid(True, alpha=0.3)

# Plot 3: CTDE + H-MARL bonuses
ax = axes[0, 2]
ax.plot(range(len(smooth(reward_log['ctde_bonus']))), smooth(reward_log['ctde_bonus']), color='#e67e22', lw=2, label='CTDE Coordination')
ax.plot(range(len(smooth(reward_log['skill_bonus']))), smooth(reward_log['skill_bonus']), color='#8e44ad', lw=2, label='H-MARL Reasoning')
ax.set_title('CTDE + H-MARL Bonuses', fontweight='bold'); ax.legend(); ax.grid(True, alpha=0.3)

# Plot 4: Early vs Late boxplot
ax = axes[1, 0]
n_cmp = min(50, len(raw) // 2)
if n_cmp > 0:
    bp = ax.boxplot([raw[:n_cmp], raw[-n_cmp:]], labels=['First 50', 'Last 50'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#e74c3c'); bp['boxes'][1].set_facecolor('#2ecc71')
ax.set_title('Early vs Late', fontweight='bold'); ax.grid(True, alpha=0.3, axis='y')

# Plot 5: Cumulative
ax = axes[1, 1]
cum = np.cumsum(raw)
ax.fill_between(range(len(cum)), cum, alpha=0.3, color='#3498db')
ax.plot(cum, color='#2c3e50', lw=2)
ax.set_title('Cumulative Reward', fontweight='bold'); ax.grid(True, alpha=0.3)

# Plot 6: α-Curriculum difficulty heatmap (NOW reflects live updates from training)
ax = axes[1, 2]
stats = curriculum.get_curriculum_stats()
tids = list(stats['task_stats'].keys())
alpha_rewards = [stats['task_stats'][t]['alpha_reward'] for t in tids]
bar_colors = ['#2ecc71' if r > -0.15 else '#e74c3c' for r in alpha_rewards]
ax.barh(tids, alpha_rewards, color=bar_colors)
ax.axvline(x=0, color='black', lw=0.5)
ax.set_title(f'\u03b1-Curriculum (\u03b1={stats["alpha"]}) [LIVE]', fontweight='bold')
ax.set_xlabel('\u03b1-Reward (closer to 0 = frontier)')

plt.tight_layout()

# ===== SAVE TO REPO =====
CURVE_PATH = os.path.join(REPO_DIR, "grpo_reward_curves.png")
fig.savefig(CURVE_PATH, dpi=150, bbox_inches='tight')
fig.savefig("grpo_reward_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"\n\u2705 Saved: {CURVE_PATH}")
print(f"   git add grpo_reward_curves.png && git commit -m 'GRPO reward curves'")

# Summary
if raw:
    print(f"\n{'='*60}")
    print("TRAINING SUMMARY")
    print(f"{'='*60}")
    print(f"  Steps: {len(raw)}")
    print(f"  First 10 avg: {np.mean(raw[:10]):.4f}")
    print(f"  Last 10 avg:  {np.mean(raw[-10:]):.4f}")
    print(f"  Improvement:  {np.mean(raw[-10:]) - np.mean(raw[:10]):+.4f}")
    print(f"  CTDE training steps: {ctde.critic.get_training_stats()}")
    print(f"  \u03b1-Curriculum (post-training): optimal_diff={curriculum.get_optimal_difficulty():.3f}")
    for k, l in zip(signal_keys + ['ctde_bonus', 'skill_bonus'],
                    signal_labels + ['CTDE Coord', 'H-MARL Reasoning']):
        v = reward_log[k]
        if len(v) >= 10:
            d = np.mean(v[-10:]) - np.mean(v[:10])
            print(f"  {l:20s} {np.mean(v[:10]):.3f} \u2192 {np.mean(v[-10:]):.3f}  ({d:+.3f})")

/tmp/ipykernel_713/3252260195.py:39: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot([raw[:n_cmp], raw[-n_cmp:]], labels=['First 50', 'Last 50'], patch_artist=True)



✅ Saved: incident-response-env/grpo_reward_curves.png
   git add grpo_reward_curves.png && git commit -m 'GRPO reward curves'

TRAINING SUMMARY
  Steps: 100
  First 10 avg: 0.4036
  Last 10 avg:  0.3551
  Improvement:  -0.0484
  CTDE training steps: {'mean_value': 0.47088749999999996, 'mean_coordination': 0.6552083333333333, 'value_trend': [0.48, 0.48, 0.05, 0.48, 0.48, 0.48, 0.48, 0.48, 0.48, 0.48], 'coordination_trend': [0.6666666666666666, 0.6666666666666666, 0.0, 0.6666666666666666, 0.6666666666666666, 0.6666666666666666, 0.6666666666666666, 0.6666666666666666, 0.6666666666666666, 0.6666666666666666]}
  α-Curriculum (post-training): optimal_diff=0.500
  Format/Phase         0.861 → 0.884  (+0.024)
  Evidence/IOC         0.013 → 0.015  (+0.002)
  Severity             0.811 → 0.699  (-0.111)
  Containment          0.128 → 0.137  (+0.009)
  Efficiency           0.450 → 0.070  (-0.380)
  Category             0.500 → 0.402  (-0.098)
  CTDE Coord           0.051 → 0.051  (-0.000)
  H-MA

## 8. 🤖 EUREKA Reflective Reward Refinement

We run the **full EUREKA feedback loop** using the trajectories collected during GRPO training. This demonstrates the closed loop: generate → evaluate → reflect → refine.

In [10]:
# Initialize EUREKA with the trajectory data collected during GRPO
eureka = EurekaRewardDesigner(llm_client=None)  # template fallback (no API needed)
eureka.set_context(
    env_source_code="IncidentResponseEnvEnvironment with 6 difficulty tasks, multi-agent SOC team",
    task_description="Train SOC analyst to investigate alerts, classify severity, execute containment, submit report",
)

# Use last 50 trajectories
recent_trajs = _trajectory_store[-50:] if len(_trajectory_store) >= 50 else _trajectory_store
recent_scores = _grader_scores[-50:] if len(_grader_scores) >= 50 else _grader_scores

# Analyze trajectories
analyzer = TrajectoryAnalyzer()
stats = analyzer.analyze_trajectories(recent_trajs)

print("\u2705 EUREKA Trajectory Analysis")
print(f"   Episodes analyzed: {stats.num_trajectories}")
print(f"   Mean score: {stats.mean_score:.3f} \u00b1 {stats.score_std:.3f}")
print(f"   Timeout rate: {stats.timeout_rate:.1%}")
print(f"   Report rate: {stats.report_rate:.1%}")
print(f"   Severity accuracy: {stats.severity_accuracy:.1%}")
if stats.common_failure_modes:
    print(f"   Failure modes: {stats.common_failure_modes}")

# Run EUREKA refinement loop (3 iterations)
print("\n\u23f3 Running EUREKA refinement loop (3 iterations)...")
best_reward = eureka.run_refinement_loop(
    trajectories=recent_trajs,
    grader_scores=recent_scores,
    n_iterations=3,
    n_candidates_per_iter=4,
    kg_stats={"evidence_count": 5, "corroboration_rate": 1.5, "source_diversity": 3},
    red_team_stats={"attack_success_rate": 0.2, "stealth_ratio": 0.6, "decoy_iocs_deployed": 2},
    overseer_violations=[{"severity": "warning", "agent_id": "l1", "description": "premature classification"}],
)

print(f"\n\u2705 EUREKA refinement complete")
print(f"   Best reward candidate: {best_reward.candidate_id if best_reward else 'N/A'}")
print(f"   Score: {best_reward.score:.4f}" if best_reward else "")
print(f"\n   Refinement history:")
for h in eureka.get_refinement_history():
    print(f"     Iter {h['iteration']}: {h['candidates']} candidates, best={h['best_score']:.4f} ({h['best_id']})")

# Show reflection
reflection = eureka.reflect_and_analyze(recent_trajs,
    kg_stats={"evidence_count": 5, "corroboration_rate": 1.5, "source_diversity": 3},
    red_team_stats={"attack_success_rate": 0.2, "stealth_ratio": 0.6},
)
print(f"\n{reflection[:800]}")

✅ EUREKA Trajectory Analysis
   Episodes analyzed: 50
   Mean score: 0.247 ± 0.078
   Timeout rate: 0.0%
   Report rate: 94.0%
   Severity accuracy: 0.0%

⏳ Running EUREKA refinement loop (3 iterations)...

✅ EUREKA refinement complete
   Best reward candidate: reward_v0_1
   Score: 0.5871

   Refinement history:
     Iter 0: 4 candidates, best=0.5871 (reward_v0_1)
     Iter 1: 3 candidates, best=0.5871 (reward_v0_1)
     Iter 2: 3 candidates, best=0.5871 (reward_v0_1)

=== EUREKA REFLECTION (Iteration 3) ===

Trajectory Analysis (50 episodes):
  Mean score: 0.247
  Score std: 0.078
  Mean steps used: 10.0
  Timeout rate: 0.0%

Action Distribution (top 5):
  - query_logs: 50
  - check_threat_intel: 50
  - classify_severity: 50
  - contain_threat: 50
  - submit_report: 50

Knowledge Graph Insights:
  Avg evidence nodes: 5
  Avg corroboration rate: 1.5
  Source diversity: 3

Red Team Impact:
  Attack success rate: 20.0%
  Stealth ratio: 60.0%
  Decoys deployed: 0

Suggested Reward Improv

## 9. 🛡️ Multi-Agent SOC Demo: L1→L2→L3 + ToM + KG + Overseer

Demonstrate the trained model working within the **full multi-agent pipeline** — including ToM belief prediction, KG evidence tracking, L1→L2→L3 handoff, and Overseer monitoring.

In [11]:
FastLanguageModel.for_inference(model)

def generate_response(alert_text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Analyze this alert:\n\n{alert_text}"},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    output = model.generate(**inputs, max_new_tokens=1500, temperature=0.7, top_p=0.9)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

# Pick the frontier task from live α-Curriculum (reflects training updates)
post_train_order = curriculum.get_curriculum_order(TASK_IDS)
demo_tid = post_train_order[0]
demo_gt = GROUND_TRUTH[demo_tid]
print(f"Demo scenario: {demo_tid} (difficulty={demo_gt['difficulty']})")
print(f"Post-training \u03b1-Curriculum order: {post_train_order}")
print(f"="*60)

# --- Initialize multi-agent team ---
l1 = L1TriageAgent()
l2 = L2SeniorAnalyst()
l3 = L3IRLead()
overseer = OverseerAgent()
board = SharedInvestigationBoard()

l1.reset(["l2_senior", "l3_lead"])
l2.reset(["l1_triage", "l3_lead"])
l3.reset(["l1_triage", "l2_senior"])
overseer.reset()

# --- H-MARL: Select initial skill ---
h_policy = HierarchicalPolicy("l1_triage", DEFAULT_SOC_SKILLS)
initial_obs = {"evidence_collected": [], "severity_set": False, "red_team_alerts": []}
selected_skill = h_policy.select_skill(initial_obs, step=1, max_steps=20)
print(f"\nH-MARL selected skill: {selected_skill.name}")
print(f"   Relevant actions: {selected_skill.relevant_actions}")

# --- L1: Initial triage (generate with GRPO-trained model) ---
print(f"\n--- L1 Triage Agent (ToM-0) ---")
alert = demo_gt["initial_observation"] or demo_gt["alert"]
l1_response = generate_response(alert)
print(f"Response (first 400 chars): {l1_response[:400]}...")

# L1 processes observation into KG
ioc_matches = re.findall(r'<iocs>(.+?)</iocs>', l1_response, re.DOTALL)
if ioc_matches:
    for ioc in ioc_matches[0].split(","):
        ioc = ioc.strip()
        if ioc:
            l1.memory.add_evidence(ioc, "ioc", source_log="model_output", confidence=0.8, is_ioc=True, step=1)

# L1 ToM + KG summary
l1_kg = l1.memory.get_belief_summary()
l1_tom = l1.tom.get_tom_summary()
print(f"\nL1 KG: {l1_kg['total_nodes']} nodes, {l1_kg['iocs_found']} IOCs")
print(f"L1 ToM: {l1_tom['dominant_level']}, weights={l1_tom['weights']}")

# Overseer monitors L1
overseer_result = overseer.monitor_action("l1_triage", "examine_alert", {}, step=1, board=board)
if overseer_result:
    print(f"Overseer violation: {overseer_result.description}")

escalate = l1.should_escalate_to_l2()
print(f"\nL1 escalate to L2? {escalate}")

# --- L1 sends message to L2 (ReSCOM) ---
msg_to_l2 = l1.prepare_message("l2_senior", message_type="handoff", step=1)
board.publish_evidence("l1_triage", l1.memory.publish_to_shared_board(threshold=0.5))
print(f"L1 \u2192 L2 message: priority={msg_to_l2.priority}, evidence={len(msg_to_l2.content.get('evidence', {}))}")

# --- L2: Deep investigation ---
print(f"\n--- L2 Senior Analyst (ToM-1) ---")
l2.receive_messages([msg_to_l2], board)
uninvestigated = l2.get_uninvestigated_leads(l1_tom)
l2_kg = l2.memory.get_belief_summary()
l2_tom = l2.tom.get_tom_summary()
print(f"L2 KG: {l2_kg['total_nodes']} nodes (merged from L1)")
print(f"L2 ToM: {l2_tom['dominant_level']}")
print(f"L2 uninvestigated leads: {uninvestigated[:3]}")
escalate_l3 = l2.should_escalate_to_l3()
print(f"L2 escalate to L3? {escalate_l3}")

# --- L3: Decision + Report ---
print(f"\n--- L3 IR Lead (ToM-2) ---")
msg_to_l3 = l2.prepare_message("l3_lead", message_type="handoff", step=2)
l3.receive_messages([msg_to_l3], board)
l3_kg = l3.memory.get_belief_summary()
l3_tom = l3.tom.get_tom_summary()
print(f"L3 KG: {l3_kg['total_nodes']} nodes (merged from L1+L2)")
print(f"L3 ToM: {l3_tom['dominant_level']}")

# L3 decoy assessment
for ioc_node in l3.memory.get_iocs()[:2]:
    decoy_prob = l3.assess_decoy_probability(ioc_node.entity)
    print(f"   Decoy probability for '{ioc_node.entity}': {decoy_prob:.2f}")

# L3 generates report from KG
report = l3.generate_report_from_kg()
print(f"\nL3 Report:\n{report[:500]}")

# --- Overseer aggregated view ---
overseer.aggregate_knowledge_graphs({
    "l1_triage": l1.memory, "l2_senior": l2.memory, "l3_lead": l3.memory
})
oversight = overseer.get_oversight_summary()
print(f"\n--- Overseer (Fleet AI) ---")
print(f"   Actions monitored: {oversight['total_actions_monitored']}")
print(f"   Violations: {oversight['violations_detected']}")
print(f"   Unified KG nodes: {oversight['unified_kg_nodes']}")
intervention = overseer.get_intervention_prompt()
if intervention:
    print(f"   Intervention: {intervention}")
else:
    print(f"   No intervention needed \u2705")

# --- Compare with ground truth via KG ---
comparison = l3.memory.compare_with_ground_truth(
    true_evidence=set(demo_gt['critical_evidence']),
    true_iocs=set(demo_gt['iocs']),
)
print(f"\nGround Truth Comparison:")
print(f"   Evidence recall: {comparison['evidence_recall']:.2f}")
print(f"   IOC recall: {comparison['ioc_recall']:.2f}")

Demo scenario: medium_hard (difficulty=0.5)
Post-training α-Curriculum order: ['medium_hard', 'medium', 'expert', 'hard_plus', 'easy', 'hard']

H-MARL selected skill: Lead Handoff
   Relevant actions: ['examine_alert', 'query_logs', 'check_threat_intel']

--- L1 Triage Agent (ToM-0) ---


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


Response (first 400 chars): <think>
The EDR alert indicates a coordinated file encryption across multiple servers, which is consistent with an active ransomware deployment. The presence of ransom notes and the use of PsExec for lateral movement from an administrative workstation further supports this conclusion. 

**Evidence Observations:**
1. **File Encryption Activity:** The EDR logs show a pattern of coordinated file encr...

L1 KG: 5 nodes, 5 IOCs
L1 ToM: TOM_0, weights={'tom_0': 0.333, 'tom_1': 0.333, 'tom_2': 0.333}

L1 escalate to L2? True
L1 → L2 message: priority=0.8, evidence=5

--- L2 Senior Analyst (ToM-1) ---
L2 KG: 5 nodes (merged from L1)
L2 ToM: TOM_0
L2 uninvestigated leads: []
L2 escalate to L3? False

--- L3 IR Lead (ToM-2) ---
L3 KG: 5 nodes (merged from L1+L2)
L3 ToM: TOM_0

L3 Report:
INCIDENT REPORT (Generated by L3 IR Lead)

EVIDENCE CHAIN:
  Step 2: [evidence] 1234 (confidence: 0.65) from peer:l2_senior
  Step 2: [evidence] 5678 (confidence: 0.65) from peer:l2_

## 10. 🛡️ Red Team Adversarial Validation

The **actual `RedTeamAgent`** from `red_team/agent.py` injects false logs, deploys decoy IOCs, covers tracks, and attempts exfiltration — testing whether the GRPO-trained model is robust.

In [12]:
red_team = RedTeamAgent()

ADVERSARIAL_TESTS = [
    {
        "name": "Inject false log + deploy decoy IOC",
        "alert": GROUND_TRUTH.get("hard", GROUND_TRUTH[TASK_IDS[2]])["initial_observation"] or GROUND_TRUTH[TASK_IDS[2]]["alert"],
        "expected_severity": GROUND_TRUTH.get("hard", GROUND_TRUTH[TASK_IDS[2]])["severity"],
        "task_id": "hard",
        "red_actions": [
            RedTeamAction(action_type=RedTeamActionType.INJECT_FALSE_LOG, target="edr",
                          payload="Benign process update \u2014 svchost.exe auto-update from WSUS. No action needed."),
            RedTeamAction(action_type=RedTeamActionType.DEPLOY_DECOY_IOC, payload="10.0.0.1"),
            RedTeamAction(action_type=RedTeamActionType.COVER_TRACKS, target="lateral_movement_log"),
        ],
    },
    {
        "name": "Stealth exfiltration attempt",
        "alert": GROUND_TRUTH.get("expert", GROUND_TRUTH[TASK_IDS[-1]])["initial_observation"] or GROUND_TRUTH[TASK_IDS[-1]]["alert"],
        "expected_severity": GROUND_TRUTH.get("expert", GROUND_TRUTH[TASK_IDS[-1]])["severity"],
        "task_id": "expert",
        "red_actions": [
            RedTeamAction(action_type=RedTeamActionType.LATERAL_MOVE_SILENTLY),
            RedTeamAction(action_type=RedTeamActionType.LATERAL_MOVE_SILENTLY),
            RedTeamAction(action_type=RedTeamActionType.EXFILTRATE_DATA),
        ],
    },
    {
        "name": "Full deception campaign",
        "alert": GROUND_TRUTH.get("easy", GROUND_TRUTH[TASK_IDS[0]])["initial_observation"] or GROUND_TRUTH[TASK_IDS[0]]["alert"],
        "expected_severity": GROUND_TRUTH.get("easy", GROUND_TRUTH[TASK_IDS[0]])["severity"],
        "task_id": "easy",
        "red_actions": [
            RedTeamAction(action_type=RedTeamActionType.INJECT_FALSE_LOG, target="auth",
                          payload="CRITICAL: Admin account compromised from 192.168.1.1 \u2014 immediate action required!"),
            RedTeamAction(action_type=RedTeamActionType.DEPLOY_DECOY_IOC, payload="evil-malware.xyz"),
            RedTeamAction(action_type=RedTeamActionType.ACCELERATE_ATTACK),
            RedTeamAction(action_type=RedTeamActionType.INJECT_FALSE_LOG, target="dns",
                          payload="DNS query for command-and-control.evil from 10.0.0.5"),
        ],
    },
]

print("="*60)
print("RED TEAM ADVERSARIAL VALIDATION")
print("="*60)

for test in ADVERSARIAL_TESTS:
    red_team.reset()
    evidence_keys = list(GROUND_TRUTH.get(test["task_id"], {}).get("critical_evidence", ["evidence_1", "evidence_2"]))
    defender_state = {"log_sources_queried": set(), "containment_score": 0.0,
                      "severity_set": False, "steps_remaining": 20}

    # Execute red team actions
    for ra in test["red_actions"]:
        red_team.step(ra, defender_state, evidence_keys)

    # Build poisoned alert
    augmented = test["alert"]
    for fl in red_team.get_false_logs():
        augmented += f"\n[Log \u2014 {fl['source']}]: {fl['content']}"
    for decoy in red_team.get_decoy_iocs():
        augmented += f"\n[ThreatIntel]: Suspicious IOC: {decoy}"

    response = generate_response(augmented)

    sev_match = re.search(r'<severity>\s*(\w+)\s*</severity>', response, re.IGNORECASE)
    predicted = sev_match.group(1).lower() if sev_match else "unknown"
    expected = test["expected_severity"].lower()
    correct = predicted == expected

    red_summary = red_team.get_reward_summary()

    icon = "\u2705" if correct else "\u274c"
    print(f"\n{icon} {test['name']}")
    print(f"   Expected: {expected}  |  Predicted: {predicted}")
    print(f"   Red Team total reward: {red_summary['total_reward']:.3f}")
    print(f"   Stealth ratio: {red_summary['stealth_ratio']:.1%}")
    print(f"   Exfiltration: {'SUCCEEDED' if red_summary['exfiltration_succeeded'] else 'BLOCKED'}")
    print(f"   False logs injected: {len(red_team.get_false_logs())}")
    print(f"   Decoy IOCs: {red_team.get_decoy_iocs()}")
    print(f"   Covered evidence: {red_team.get_covered_evidence()}")

    # Update curriculum with adversarial result
    curriculum.record_performance(test["task_id"], 1.0 if correct else 0.0)

print(f"\n\u03b1-Curriculum updated with adversarial results:")
print(f"   New optimal difficulty: {curriculum.get_optimal_difficulty():.3f}")

RED TEAM ADVERSARIAL VALIDATION

✅ Inject false log + deploy decoy IOC
   Expected: critical  |  Predicted: critical
   Red Team total reward: 0.280
   Stealth ratio: 66.7%
   Exfiltration: BLOCKED
   False logs injected: 1
   Decoy IOCs: ['10.0.0.1']
   Covered evidence: ['lateral_movement_log']

✅ Stealth exfiltration attempt
   Expected: critical  |  Predicted: critical
   Red Team total reward: 1.050
   Stealth ratio: 66.7%
   Exfiltration: SUCCEEDED
   False logs injected: 0
   Decoy IOCs: []
   Covered evidence: []


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



❌ Full deception campaign
   Expected: high  |  Predicted: critical
   Red Team total reward: 0.280
   Stealth ratio: 75.0%
   Exfiltration: BLOCKED
   False logs injected: 2
   Decoy IOCs: ['evil-malware.xyz']
   Covered evidence: []

α-Curriculum updated with adversarial results:
   New optimal difficulty: 0.500


## 11. 💾 Save & Export

In [ ]:
model.save_pretrained("grpo_soc_lora")
tokenizer.save_pretrained("grpo_soc_lora")
print("\u2705 LoRA saved")

model.save_pretrained_merged("soc_reasoning_model_16bit", tokenizer, save_method="merged_16bit")
print("\u2705 16-bit merged model saved")

model.save_pretrained_gguf("soc_reasoning_model_gguf", tokenizer, quantization_method="q4_k_m")
print("\u2705 GGUF Q4_K_M saved")

# Final stats from every subsystem
print("\n" + "="*60)
print("FINAL SUBSYSTEM STATUS")
print("="*60)
print(f"  \u03b1-Curriculum:  {json.dumps(curriculum.get_curriculum_stats(), indent=4, default=str)[:500]}")
print(f"  CTDE:          {json.dumps(ctde.get_training_summary(), indent=4, default=str)[:300]}")
print(f"  H-MARL skills: {json.dumps(skill_discovery.get_skill_summary()[:3], indent=4)}")
print(f"  EUREKA iters:  {len(eureka.iterations)}")
print(f"  Red Team tests: {len(ADVERSARIAL_TESTS)}")
print(f"  Overseer:      {json.dumps(overseer.get_oversight_summary(), indent=4)}")
print(f"  Reward curve:  {CURVE_PATH}")

print("\n\u2705 All done. To commit:")
print(f"  cd {REPO_DIR}")
print(f"  git add grpo_reward_curves.png")
print(f"  git commit -m 'Add GRPO training artifacts'")

Unsloth: Restored added_tokens_decoder metadata in grpo_soc_lora/tokenizer_config.json.


✅ LoRA saved


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in soc_reasoning_model_16bit/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.97G [00:00<?, ?B/s]

---

## Architecture Summary

```
tasks/ (Scenario dataclasses)
  │
  ├─ Ground truth: severity, category, IOCs, containment
  └─ Alert text for GRPO prompts
       │
       ▼
  AlphaCurriculumSelector (LIVE throughout training)
  ├─ Orders tasks by learning frontier (P(success)≈α)
  ├─ Weights frontier tasks in dataset
  └─ Updated every training step with actual rollout rewards
       │
       ▼
  GRPO Training (200 steps)
  ├─ 6 reward fns (env-grader aligned)
  ├─ + CentralizedCritic coordination proxy (CTDE)
  ├─ + Reasoning-depth & hypothesis-diversity bonus (H-MARL)
  └─ Trajectory storage for EUREKA
       │
       ▼
  EUREKA Reflective Refinement
  ├─ TrajectoryAnalyzer (failure modes, action dist)
  ├─ generate → evaluate → reflect → refine loop
  └─ Fed by KG stats + Red Team stats + Overseer violations
       │
       ▼
  Multi-Agent Demo (L1→L2→L3)
  ├─ A-ToM: Hedge algorithm, belief prediction
  ├─ DAMCS KG: evidence chains, auto-linking, decoy flags
  ├─ ReSCOM: SharedInvestigationBoard, messages
  ├─ Overseer: policy monitoring, safety checks, unified KG
  └─ H-MARL: skill selection, filtered actions
       │
       ▼
  RedTeamAgent Adversarial Validation
  ├─ inject_false_log, deploy_decoy_ioc
  ├─ cover_tracks, accelerate_attack
  ├─ lateral_move_silently, exfiltrate_data
  └─ Reward signal: visible per step
       │
       ▼
  grpo_reward_curves.png + saved model
```

### All 9 Features Used

| Feature | Where Used | Cell |
|---|---|---|
| α-Curriculum | Dataset ordering + LIVE updates during training | 4, 6 |
| Red Team | Adversarial validation (3 scenarios) | 10 |
| A-ToM | L1/L2/L3 ToM levels + Hedge weights | 9 |
| DAMCS KG | Evidence tracking, decoy detection, reporting | 9 |
| ReSCOM | L1\u2192L2 handoff messages + shared board | 9 |
| Overseer | Action monitoring + unified KG | 9 |
| EUREKA | Trajectory analysis + refinement loop | 8 |
| CTDE | Coordination proxy in reward function | 6 |
| H-MARL | Reasoning-depth + hypothesis-diversity bonus | 6, 9 |